# 01. Stage 1: No-Pandemic Counterfactual Forecasts

**Geographic vintage:** September 2018 OMB. **Information cutoff:**
February 29, 2020. The last full QCEW quarter published by then is 2019 Q2.
Consequently, **model fitting, tuning, and selection stop at June 2019**.

Each CBSA independently evaluates a prespecified grid of seasonal naive,
seasonal drift, log-linear trend-seasonal, ETS structures, and SARIMA orders.
Three 12-month rolling-origin holdouts, ending by June 2019, determine the
lowest mean normalized MAE. The winning specification is refitted through
June 2019, then extrapolated from July 2019 through December 2023. Realized
2020–2023 employment is joined *only afterward* to calculate gaps.

**Caution:** The winning cross-validation score is a selection score and may be
optimistic because the same pre-pandemic folds tune candidate specifications.
July 2019–February 2020 observations form a separate post-selection diagnostic
window and are never used for fitting or selection. Long-horizon forecasts are
extrapolations, not identified causal effects. Downloaded historical QCEW files
may reflect revisions made after February 2020; archived release-vintage data
would be required for an exact real-time reconstruction.

In [3]:
from pathlib import Path
import os, json, hashlib, warnings, time, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.linear_model import LinearRegression
from threadpoolctl import threadpool_limits

ROOT = (Path(os.environ['RL_PROJECT_ROOT']).expanduser().resolve()
        if os.environ.get('RL_PROJECT_ROOT') else next(
            (p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'data'/'processed'/'omb2018_geography_manifest.json').is_file()), None))
if ROOT is None:
    raise FileNotFoundError('Run notebook 00 first (2018 OMB panel and manifest).')
PROC, OUT = ROOT/'data'/'processed', ROOT/'output'
OUT.mkdir(parents=True, exist_ok=True)
GEO = json.loads((PROC/'omb2018_geography_manifest.json').read_text())
if GEO['geography_vintage'] != '2018-09':
    raise ValueError('Expected September 2018 OMB delineations; refuse mixed vintages.')
PANEL = PROC/'qcew_cbsa_monthly_2010_2023_omb2018.csv'

def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as handle:
        for b in iter(lambda: handle.read(4*1024*1024), b''):
            h.update(b)
    return h.hexdigest()

if sha256(PANEL) != GEO['panel_sha256']:
    raise ValueError('2018 panel hash differs from notebook 00 geography manifest.')
TRAIN_END = pd.Timestamp('2019-06-01')
FORECAST_END = pd.Timestamp('2023-12-01')
VALIDATION_ORIGINS = tuple(pd.Timestamp(x) for x in
                           ('2016-06-01','2017-06-01','2018-06-01'))
VALIDATION_MONTHS = 12
RUN_ETS = os.environ.get('RL_RUN_ETS','1') == '1'
RUN_SARIMA = os.environ.get('RL_RUN_SARIMA','1') == '1'
ETS_GRID = os.environ.get('RL_ETS_GRID','standard').lower()
SARIMA_GRID = os.environ.get('RL_SARIMA_GRID','standard').lower()
N_JOBS = int(os.environ.get('RL_N_JOBS','8'))
MAX_CBSAS = int(os.environ['RL_MAX_CBSAS']) if os.environ.get('RL_MAX_CBSAS') else None
SARIMA_MAXITER = int(os.environ.get('RL_SARIMA_MAXITER','80'))
if N_JOBS < 1 or (MAX_CBSAS is not None and MAX_CBSAS < 1):
    raise ValueError('RL_N_JOBS and RL_MAX_CBSAS must be positive.')
if ETS_GRID not in ('fast','standard'):
    raise ValueError('RL_ETS_GRID must be fast or standard')
if SARIMA_GRID not in ('smoke','fast','standard','extended'):
    raise ValueError('RL_SARIMA_GRID must be smoke, fast, standard, or extended')
np.random.seed(2026)
print('Project:',ROOT)
print('2018 OMB geography | as-of: 2020-02-29 | latest fitting month:',TRAIN_END.date())
print('ETS:',RUN_ETS,ETS_GRID,'SARIMA:',RUN_SARIMA,SARIMA_GRID,'workers:',N_JOBS)
print('To test quickly: RL_MAX_CBSAS=2, RL_N_JOBS=1, RL_SARIMA_GRID=smoke')

Project: c:\Users\adamt\OneDrive\Documents\Directed Practicum\regional-labor-resilience
2018 OMB geography | as-of: 2020-02-29 | latest fitting month: 2019-06-01
ETS: True standard SARIMA: True standard workers: 8
To test quickly: RL_MAX_CBSAS=2, RL_N_JOBS=1, RL_SARIMA_GRID=smoke


## 1. Load the verified 2018 OMB panel

Stage 1 outcomes may extend through 2023, but every model-fitting call below
receives **only** the June 2019-and-earlier slice.

In [4]:
panel = pd.read_csv(PANEL, dtype={'cbsa_code':'string'},parse_dates=['date'])
required = {'cbsa_code','date','employment','cbsa_type'}
if not required.issubset(panel):
    raise ValueError(f'Missing panel fields: {sorted(required-set(panel))}')
panel['cbsa_code'] = panel.cbsa_code.astype('string').str.zfill(5)
panel['employment'] = pd.to_numeric(panel.employment,errors='coerce')
if panel.duplicated(['cbsa_code','date']).any():
    raise ValueError('Duplicate CBSA and month')
if panel.employment.isna().any() or (panel.employment <= 0).any():
    raise ValueError('Missing or nonpositive employment in the Stage 1 panel')
if panel.cbsa_code.nunique() != GEO['n_cbsa']:
    raise ValueError('CBSA count changed since notebook 00')
months = pd.date_range('2010-01-01','2023-12-01',freq='MS')
if not panel.groupby('cbsa_code').date.nunique().eq(len(months)).all():
    raise ValueError('Incomplete monthly CBSA panel')
pre = panel.loc[panel.date <= TRAIN_END].copy()
cbsa_ids = sorted(pre.cbsa_code.unique())
if MAX_CBSAS:
    cbsa_ids = cbsa_ids[:MAX_CBSAS]
tasks = [(cbsa, group.sort_values('date').employment.to_numpy(dtype=float))
         for cbsa,group in pre.loc[pre.cbsa_code.isin(cbsa_ids)].groupby('cbsa_code',sort=True)]
expected_pre_dates = pd.date_range('2010-01-01',TRAIN_END,freq='MS')
if any(len(y) != len(expected_pre_dates) for _,y in tasks):
    raise ValueError('Incomplete pre-cutoff employment series')
print('CBSAs:',len(tasks),'out of',GEO['n_cbsa'])

CBSAs: 936 out of 936


## 2. Define candidate forecasting specifications

ETS candidates vary trend,
damping and seasonal form. SARIMA candidates vary both nonseasonal and
seasonal orders, with annual (12-month) seasonality and a log transform.

Choose `RL_SARIMA_GRID=fast` for an initial national-scale run, `standard`
(default) for the primary analysis, or `extended` for a sensitivity analysis.
Larger grids are much slower: approximate SARIMA fits per CBSA =
`number of grid candidates × 3 validation origins + 1 final refit`.

In [ ]:
def seasonal_naive(y,h):
    return np.tile(np.asarray(y[-12:],float),int(np.ceil(h/12)))[:h]

def seasonal_drift(y,h):
    y=np.asarray(y,float)
    years=(len(y)-12)/12
    annual_change=(y[-12:].mean()-y[:12].mean())/years
    return seasonal_naive(y,h) + annual_change*((np.arange(h)+1)/12)

def trend_seasonal(y,h):
    y=np.asarray(y,float)
    train_t=np.arange(len(y))
    test_t=np.arange(len(y),len(y)+h)
    def design(t):
        # Monthly level (not seasonal-difference) deterministic log trend.
        return np.column_stack([t]+[(t%12==k).astype(float) for k in range(1,12)])
    fit=LinearRegression().fit(design(train_t),np.log(y))
    pred_log=fit.predict(design(test_t))
    if not np.isfinite(pred_log).all() or np.abs(pred_log).max()>100:
        raise ValueError('Unstable trend-seasonal extrapolation')
    return np.exp(pred_log)

BASELINES = [
    {'family':'seasonal_naive','spec':'seasonal_naive','config':{}},
    {'family':'seasonal_drift','spec':'seasonal_drift','config':{}},
    {'family':'trend_seasonal','spec':'trend_seasonal_log','config':{}},
]

ets_options=[
    {'trend':None,'seasonal':'add','damped_trend':False},
    {'trend':'add','seasonal':'add','damped_trend':False},
    {'trend':'add','seasonal':'add','damped_trend':True},
    {'trend':None,'seasonal':'mul','damped_trend':False},
    {'trend':'add','seasonal':'mul','damped_trend':False},
    {'trend':'add','seasonal':'mul','damped_trend':True},
]
if ETS_GRID=='fast':
    ets_options=ets_options[:3]

sarima_options = [
    ((0,1,1),(0,1,1,12)),
    ((1,1,0),(0,1,1,12)),
    ((1,1,1),(0,1,1,12)),
    ((0,1,1),(1,1,0,12)),
    ((1,1,0),(1,1,0,12)),
    ((1,1,1),(1,1,0,12)),
    ((0,1,1),(1,1,1,12)),
    ((1,1,0),(1,1,1,12)),
    ((1,1,1),(1,1,1,12)),
    ((0,1,1),(0,0,1,12)),
    ((1,1,0),(0,0,1,12)),
    ((1,1,1),(0,0,1,12)),
    ((0,1,1),(1,0,0,12)),
    ((1,1,0),(1,0,0,12)),
    ((1,1,1),(1,0,0,12)),
]
if SARIMA_GRID=='smoke':
    sarima_options=sarima_options[:2]
elif SARIMA_GRID=='fast':
    sarima_options=sarima_options[:6]
elif SARIMA_GRID=='extended':
    sarima_options += [
        ((0,0,1),(0,1,1,12)), ((1,0,0),(0,1,1,12)),
        ((1,0,1),(0,1,1,12)), ((0,0,1),(1,1,0,12)),
        ((1,0,0),(1,1,0,12)), ((1,0,1),(1,1,0,12)),
        ((0,0,1),(0,0,1,12)), ((1,0,0),(0,0,1,12)),
        ((1,0,1),(0,0,1,12)), ((2,1,0),(0,1,1,12)),
        ((0,1,2),(0,1,1,12)), ((2,1,0),(1,1,0,12)),
        ((0,1,2),(1,1,0,12)),
    ]

candidates=list(BASELINES)
if RUN_ETS:
    for conf in ets_options:
        spec=f"ETS_trend-{conf['trend'] or 'none'}_season-{conf['seasonal']}_damped-{int(conf['damped_trend'])}"
        candidates.append({'family':'ets','spec':spec,'config':conf})
if RUN_SARIMA:
    for order,seasonal_order in sarima_options:
        spec=(f"SARIMA_{''.join(map(str,order))}_"
              f"{''.join(map(str,seasonal_order[:3]))}_s12_log")
        candidates.append({'family':'sarima','spec':spec,
                           'config':{'order':list(order),'seasonal_order':list(seasonal_order)}})
assert len(set(x['spec'] for x in candidates)) == len(candidates)
print('Candidate specifications:',len(candidates),'(ETS:',len(ets_options) if RUN_ETS else 0,
      '| SARIMA:',len(sarima_options) if RUN_SARIMA else 0,')')

def fit_forecast(y, h, candidate):
    # Crucial: y is provided as a pre-cutoff slice, not the full panel.
    family=candidate['family']; cfg=candidate['config']
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        if family=='seasonal_naive':
            pred=seasonal_naive(y,h)
        elif family=='seasonal_drift':
            pred=seasonal_drift(y,h)
        elif family=='trend_seasonal':
            pred=trend_seasonal(y,h)
        elif family=='ets':
            from statsmodels.tsa.holtwinters import ExponentialSmoothing
            model=ExponentialSmoothing(
                np.asarray(y,float),trend=cfg['trend'],seasonal=cfg['seasonal'],
                damped_trend=cfg['damped_trend'],seasonal_periods=12,
                initialization_method='estimated')
            fitted=model.fit(optimized=True,use_brute=False)
            pred=np.asarray(fitted.forecast(h),float)
        elif family=='sarima':
            from statsmodels.tsa.statespace.sarimax import SARIMAX
            model=SARIMAX(np.log(np.asarray(y,float)),
                order=tuple(cfg['order']),seasonal_order=tuple(cfg['seasonal_order']),
                trend='n',enforce_stationarity=False,enforce_invertibility=False)
            fit=model.fit(disp=False,maxiter=SARIMA_MAXITER)
            if not fit.mle_retvals.get('converged',False):
                raise ValueError('SARIMA optimizer did not converge')
            log_pred=np.asarray(fit.forecast(h),float)
            if not np.isfinite(log_pred).all() or np.abs(log_pred).max()>100:
                raise ValueError('Nonfinite or explosive log-SARIMA prediction')
            pred=np.exp(log_pred)
        else:
            raise ValueError(f'Unknown model family: {family}')
    pred=np.asarray(pred,dtype=float).reshape(-1)
    if len(pred)!=h or not np.isfinite(pred).all() or (pred<=0).any():
        raise ValueError('Forecast has incorrect length, nonfinite, or nonpositive values')
    return pred

Candidate specifications: 24 (ETS: 6 | SARIMA: 15 )


## 3. Rolling-origin cross-validation and separate specification tuning per CBSA

Validation origins: June 2016, June 2017, June 2018; each forecast covers
the following 12 months. No fold sees data beyond June 2019. Each
individual ETS and SARIMA specification must pass *all three validation
origins* to qualify. The winning specification is the lowest mean normalized
MAE (mean absolute error divided by that fold's training mean employment).

**Computational cost:** `standard` evaluates 3 baselines + 6 ETS + 15 SARIMA
candidates × 3 folds for each region, plus one or more final fits. 

In [6]:
FORECAST_DATES = pd.date_range(TRAIN_END+pd.offsets.MonthBegin(1),FORECAST_END,freq='MS')
assert len(FORECAST_DATES)==54 and FORECAST_DATES[0]==pd.Timestamp('2019-07-01')
origin_locs=[]
for origin in VALIDATION_ORIGINS:
    train_len=len(pd.date_range('2010-01-01',origin,freq='MS'))
    if train_len+VALIDATION_MONTHS>len(expected_pre_dates):
        raise ValueError(f'Validation date {origin} overlaps post-cutoff period')
    origin_locs.append((origin,train_len))

def evaluate_cbsa(cbsa, y):
    cv_rows=[]; summary_rows=[]; failed=[]; selected=None; forecast_rows=[]
    for candidate in candidates:
        fold_metrics=[]
        for origin,ntrain in origin_locs:
            t0=time.perf_counter()
            try:
                pred=fit_forecast(y[:ntrain],VALIDATION_MONTHS,candidate)
                truth=y[ntrain:ntrain+VALIDATION_MONTHS]
                mae=float(np.abs(truth-pred).mean())
                nmae=mae/float(y[:ntrain].mean())
                if not np.isfinite(nmae):
                    raise ValueError('Invalid normalized MAE')
                cv_rows.append({'cbsa_code':cbsa,'model':candidate['family'],
                    'specification':candidate['spec'],'origin':origin.date().isoformat(),
                    'mae':mae,'normalized_mae':nmae,
                    'run_seconds':round(time.perf_counter()-t0,3)})
                fold_metrics.append(nmae)
            except Exception as exc:
                failed.append({'cbsa_code':cbsa,'model':candidate['family'],
                    'specification':candidate['spec'],'origin':origin.date().isoformat(),
                    'error':f'{type(exc).__name__}: {exc}'})
        if len(fold_metrics)==len(origin_locs):
            score=float(np.mean(fold_metrics))
            summary_rows.append({'cbsa_code':cbsa,'model':candidate['family'],
                'specification':candidate['spec'],'cv_normalized_mae':score,
                'cv_folds':len(fold_metrics),'config_json':json.dumps(candidate['config'],sort_keys=True)})

    # Candidates failing any origin are ineligible. Resolve final fit failures
    # using only the training sample and previously calculated CV scores.
    ranked=sorted(summary_rows,key=lambda row:(row['cv_normalized_mae'],row['specification']))
    specs={candidate['spec']:candidate for candidate in candidates}
    for rank,row in enumerate(ranked, start=1):
        candidate=specs[row['specification']]
        try:
            p=fit_forecast(y,len(FORECAST_DATES),candidate)
            selected={'cbsa_code':cbsa,'model':candidate['family'],
                'specification':candidate['spec'],
                'config_json':json.dumps(candidate['config'],sort_keys=True),
                'cv_normalized_mae':row['cv_normalized_mae'],
                'selected_cv_rank':rank,'training_end':TRAIN_END.date().isoformat()}
            forecast_rows=[{'cbsa_code':cbsa,'date':dt.strftime('%Y-%m-%d'),
                'counterfactual_employment':float(value),
                'selected_model':candidate['family'],
                'selected_specification':candidate['spec']}
                for dt,value in zip(FORECAST_DATES,p)]
            break
        except Exception as exc:
            failed.append({'cbsa_code':cbsa,'model':candidate['family'],
                'specification':candidate['spec'],'origin':'final',
                'error':f'{type(exc).__name__}: {exc}'})
    return selected,cv_rows,summary_rows,forecast_rows,failed

start=time.perf_counter()
if N_JOBS==1:
    results=[evaluate_cbsa(cbsa,y) for cbsa,y in tasks]
else:
    results=Parallel(n_jobs=N_JOBS,backend='loky',verbose=5)(
        delayed(evaluate_cbsa)(cbsa,y) for cbsa,y in tasks)

selected=pd.DataFrame([r[0] for r in results if r[0] is not None],
    columns=['cbsa_code','model','specification','config_json','cv_normalized_mae',
             'selected_cv_rank','training_end'])
cv=pd.DataFrame([row for r in results for row in r[1]],
    columns=['cbsa_code','model','specification','origin','mae','normalized_mae','run_seconds'])
summary=pd.DataFrame([row for r in results for row in r[2]],
    columns=['cbsa_code','model','specification','cv_normalized_mae','cv_folds','config_json'])
forecast=pd.DataFrame([row for r in results for row in r[3]],
    columns=['cbsa_code','date','counterfactual_employment','selected_model','selected_specification'])
failures=pd.DataFrame([row for r in results for row in r[4]],
    columns=['cbsa_code','model','specification','origin','error'])

# Always write audits, even if there were fit failures, so troubleshooting is possible.
cv.to_csv(OUT/'stage1_omb2018_cv_metrics.csv',index=False)
summary.to_csv(OUT/'stage1_omb2018_candidate_specification_scores.csv',index=False)
failures.to_csv(OUT/'stage1_omb2018_model_fit_failures.csv',index=False)
selected.to_csv(OUT/'stage1_omb2018_selected_models.csv',index=False)
if len(selected)!=len(tasks):
    missed=sorted(set(x for x,_ in tasks)-set(selected.cbsa_code))
    raise RuntimeError(f'No usable forecast for {len(missed)} CBSA(s); '
                       f'first missing IDs: {missed[:10]}. Inspect model_fit_failures.csv')
forecast.to_csv(PROC/'stage1_omb2018_forecasts_jul2019_dec2023.csv',index=False)
assert len(forecast)==len(tasks)*len(FORECAST_DATES)
assert not forecast.duplicated(['cbsa_code','date']).any()
print('Selected model families:')
print(selected.model.value_counts().to_string())
print('Unique SARIMA specifications chosen:',selected.loc[selected.model.eq('sarima'),'specification'].nunique())
print('Unique ETS configurations chosen:',selected.loc[selected.model.eq('ets'),'specification'].nunique())
print('Total elapsed minutes:',round((time.perf_counter()-start)/60,2))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   11.9s
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:   35.3s
[Parallel(n_jobs=8)]: Done 146 tasks      | elapsed:  1.3min
[Parallel(n_jobs=8)]: Done 272 tasks      | elapsed:  2.2min
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:  3.5min
[Parallel(n_jobs=8)]: Done 632 tasks      | elapsed:  5.0min
[Parallel(n_jobs=8)]: Done 866 tasks      | elapsed:  6.7min
[Parallel(n_jobs=8)]: Done 936 out of 936 | elapsed:  7.2min finished


Selected model families:
model
ets               503
sarima            321
trend_seasonal     56
seasonal_naive     32
seasonal_drift     24
Unique SARIMA specifications chosen: 15
Unique ETS configurations chosen: 6
Total elapsed minutes: 7.25


## 4. Hold-out diagnostic: July 2019–February 2020 (never used for fitting)

These eight later observed pre-pandemic months are *diagnostic only*.
Their measured values were not publicly eligible under the June 2019 QCEW
cutoff; they are not used to alter any selected model or hyperparameters.

In [7]:
forecast['date']=pd.to_datetime(forecast.date)
forecast_ids=set(forecast.cbsa_code)
future=panel.loc[panel.cbsa_code.isin(forecast_ids) &
                 panel.date.ge(pd.Timestamp('2020-03-01')),
                 ['cbsa_code','date','employment','cbsa_type']]
gaps=forecast.loc[forecast.date.ge(pd.Timestamp('2020-03-01'))].merge(
    future,on=['cbsa_code','date'],how='left',validate='one_to_one')
if gaps.employment.isna().any() or gaps.counterfactual_employment.isna().any():
    raise ValueError('Missing observed or predicted pandemic-period employment')
if not gaps.groupby('cbsa_code').size().eq(46).all():
    raise ValueError('Expected 46 months (Mar 2020–Dec 2023) for each CBSA')
gaps['gap_pct']=100*(gaps.employment/gaps.counterfactual_employment-1)
gaps['shortfall_pct']=(-gaps.gap_pct).clip(lower=0)
GAPS_FILE=PROC/'stage1_omb2018_observed_vs_counterfactual_2020_2023.csv'
gaps.to_csv(GAPS_FILE,index=False)

cal=forecast.loc[forecast.date.le(pd.Timestamp('2020-02-01'))].merge(
    panel.loc[panel.cbsa_code.isin(forecast_ids),['cbsa_code','date','employment']],
    on=['cbsa_code','date'],how='left',validate='one_to_one')
if len(cal)!=len(tasks)*8 or cal.employment.isna().any():
    raise ValueError('Expected eight pre-pandemic diagnostic months per CBSA')
cal['gap_pct']=100*(cal.employment/cal.counterfactual_employment-1)
cal.to_csv(OUT/'stage1_omb2018_precovid_forecast_diagnostic.csv',index=False)

agg=gaps.groupby('date',as_index=False)[['employment','counterfactual_employment']].sum()
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(agg.date,agg.employment,label='Observed covered employment')
ax.plot(agg.date,agg.counterfactual_employment,'--',label='No-pandemic extrapolation')
ax.set(title='2018 OMB CBSAs: observed employment vs counterfactual',ylabel='Covered employment')
ax.legend();fig.tight_layout()
fig.savefig(OUT/'stage1_omb2018_observed_vs_forecast.png',dpi=200)
plt.close(fig)
print('Saved Stage 1 observed-vs-counterfactual:',len(gaps),'rows')

Saved Stage 1 observed-vs-counterfactual: 43056 rows


## 5. Reproducibility manifest and handoff to Stage 2

In [9]:
manifest={
    'geography_vintage':'2018-09',
    'input_panel_sha256':sha256(PANEL),
    'information_cutoff':'2020-02-29',
    'model_training_cutoff':'2019-06-01',
    'rolling_validation_origins':[x.date().isoformat() for x in VALIDATION_ORIGINS],
    'validation_horizon_months':VALIDATION_MONTHS,
    'forecast_window':'2019-07 through 2023-12',
    'outcome_window':'2020-03 through 2023-12',
    'model_candidates':sorted(set(x['family'] for x in candidates)),
    'candidate_specification_count':len(candidates),
    'ets_grid':ETS_GRID if RUN_ETS else 'disabled',
    'sarima_grid':SARIMA_GRID if RUN_SARIMA else 'disabled',
    'sarima_maxiter':SARIMA_MAXITER,
    'cbsa_count':len(tasks),
    'all_regions_included':MAX_CBSAS is None,
    'selected_model_count':len(selected),
    'total_validation_rows':len(cv),
    'total_failed_fit_attempts':len(failures),
    'selected_model_families':selected.model.value_counts().to_dict(),
    'historical_release_vintage_verified':False,
    'note':'QCEW historical downloads may contain revisions after the real-time information cutoff. Model selection is CBSA-specific, and candidate-level cross-validation scores are not independent accuracy estimates.',
    'selected_models_sha256':sha256(OUT/'stage1_omb2018_selected_models.csv'),
    'candidate_scores_sha256':sha256(OUT/'stage1_omb2018_candidate_specification_scores.csv'),
    'outcomes_sha256':sha256(GAPS_FILE),
    'completed_utc':datetime.now(timezone.utc).isoformat(),
    'python_version':platform.python_version(),
    'pandas_version':pd.__version__,
}
(OUT/'stage1_omb2018_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Stage 1 complete.')

Stage 1 complete.
